In [ ]:
# ============================================================
# CÉLULA 1 — CARREGAMENTO DOS DADOS
# ============================================================

import pandas as pd
import numpy as np

# Escolha a quantidade de partidas: 100, 1000 ou 10000
quantidade_partidas = 1000

caminho_arquivo = f"../data/team-time-series-{quantidade_partidas}.xlsx"

df_series_temporais = pd.read_excel(caminho_arquivo)

display(df_series_temporais.head())

print(f"Arquivo carregado: {caminho_arquivo}")
print(f"Quantidade de linhas: {df_series_temporais.shape[0]}")
print(f"Quantidade de colunas: {df_series_temporais.shape[1]}")

print("\nColunas disponíveis:")
print(df_series_temporais.columns.tolist())

In [ ]:

# ============================================================
# CÉLULA 2 — ORGANIZAÇÃO DAS SÉRIES TEMPORAIS
# ============================================================

# Colunas que não serão utilizadas como características de entrada
colunas_excluidas = [
    "match_id",
    "timestep",
    "teamId",
    "vencedor"
]

colunas_features = [
    coluna
    for coluna in df_series_temporais.columns
    if coluna not in colunas_excluidas
]

# Remover o último timestep de cada partida
ultimo_timestep = (
    df_series_temporais
    .groupby("match_id")["timestep"]
    .transform("max")
)

df_sem_ultimo_timestep = df_series_temporais[
    df_series_temporais["timestep"] < ultimo_timestep
].copy()

# Dicionários para armazenar as matrizes e os respectivos alvos
partidas = {}
vencedores = {}

# Organizar as linhas por partida, instante e equipe
dados_ordenados = df_sem_ultimo_timestep.sort_values(
    ["match_id", "timestep", "teamId"]
)

for match_id, grupo in dados_ordenados.groupby("match_id", sort=False):

    # Identificar o vencedor da partida
    valores_vencedor = grupo["vencedor"].dropna().unique()

    if len(valores_vencedor) != 1:
        continue

    vencedor = valores_vencedor[0]

    if vencedor not in (100, 200):
        continue

    sequencia = []
    partida_valida = True

    for timestep, instante in grupo.groupby("timestep", sort=True):

        # Cada instante deve conter exatamente uma linha por equipe
        equipes_presentes = instante["teamId"].tolist()

        if (
            len(equipes_presentes) != 2
            or sorted(equipes_presentes) != [100, 200]
        ):
            partida_valida = False
            break

        # Garantir a ordem Blue (100) e Red (200)
        instante_ordenado = (
            instante
            .set_index("teamId")
            .reindex([100, 200])
        )

        # Matriz do instante: (2 equipes, características)
        matriz_instante = instante_ordenado[
            colunas_features
        ].to_numpy(dtype=np.float32)

        # Verificar se o instante possui dados completos
        if not np.isfinite(matriz_instante).all():
            partida_valida = False
            break

        sequencia.append(matriz_instante)

    if partida_valida and sequencia:
        # Matriz da partida: (instantes, 2 equipes, características)
        partidas[match_id] = np.stack(sequencia)
        vencedores[match_id] = int(vencedor)

# Resumo da organização
total_partidas = df_series_temporais["match_id"].nunique()

print(f"Partidas carregadas: {total_partidas}")
print(f"Partidas organizadas: {len(partidas)}")
print(f"Partidas descartadas na organização: "
      f"{total_partidas - len(partidas)}")
print(f"Características por equipe: {len(colunas_features)}")

print(f"\nLinhas removidas: "
      f"{len(df_series_temporais) - len(df_sem_ultimo_timestep)}")

if partidas:
    primeiro_match_id = next(iter(partidas))

    print(f"\nPartida de exemplo: {primeiro_match_id}")
    print(f"Formato da matriz: {partidas[primeiro_match_id].shape}")
    print(f"Vencedor (100 = Blue; 200 = Red): "
          f"{vencedores[primeiro_match_id]}")


In [ ]:
# ============================================================
# CÉLULA 3 — DIVISÃO DOS DADOS
# ============================================================

# Definir semente para reprodutibilidade
rng = np.random.default_rng(42)

# Separar partidas por equipe vencedora
ids_blue = np.array([
    match_id for match_id in partidas
    if vencedores[match_id] == 100
])

ids_red = np.array([
    match_id for match_id in partidas
    if vencedores[match_id] == 200
])

# Embaralhar as partidas de cada classe
rng.shuffle(ids_blue)
rng.shuffle(ids_red)

# Dividir cada classe em treinamento, validação e teste
def dividir_classe(ids):
    quantidade = len(ids)

    quantidade_treino = int(quantidade * 0.70)
    quantidade_validacao = int(quantidade * 0.15)

    treino = ids[:quantidade_treino]
    validacao = ids[
        quantidade_treino:quantidade_treino + quantidade_validacao
    ]
    teste = ids[quantidade_treino + quantidade_validacao:]

    return treino, validacao, teste


blue_treino, blue_validacao, blue_teste = dividir_classe(ids_blue)
red_treino, red_validacao, red_teste = dividir_classe(ids_red)

# Combinar as duas classes em cada conjunto
ids_treino = np.concatenate([blue_treino, red_treino])
ids_validacao = np.concatenate([blue_validacao, red_validacao])
ids_teste = np.concatenate([blue_teste, red_teste])

# Embaralhar os conjuntos
rng.shuffle(ids_treino)
rng.shuffle(ids_validacao)
rng.shuffle(ids_teste)

# Exibir resumo da divisão
print(f"Partidas para treinamento: {len(ids_treino)}")
print(f"Partidas para validação: {len(ids_validacao)}")
print(f"Partidas para teste: {len(ids_teste)}")

print("\nDistribuição dos vencedores:")

for nome, ids in [
    ("Treinamento", ids_treino),
    ("Validação", ids_validacao),
    ("Teste", ids_teste)
]:
    blue = sum(vencedores[mid] == 100 for mid in ids)
    red = sum(vencedores[mid] == 200 for mid in ids)

    print(f"{nome}: Blue = {blue} | Red = {red}")

# Verificar se não existem partidas compartilhadas entre os conjuntos
assert set(ids_treino).isdisjoint(ids_validacao)
assert set(ids_treino).isdisjoint(ids_teste)
assert set(ids_validacao).isdisjoint(ids_teste)

print("\nDivisão realizada sem sobreposição de partidas.")

In [ ]:
# ============================================================
# CÉLULA 4 — NORMALIZAÇÃO DOS DADOS
# ============================================================

# Reunir os dados das partidas de treinamento
dados_normalizacao = np.concatenate([
    partidas[match_id].reshape(-1, len(colunas_features))
    for match_id in ids_treino
], axis=0)

# Calcular média e desvio padrão
media_features = dados_normalizacao.mean(axis=0)
desvio_features = dados_normalizacao.std(axis=0)

# Evitar divisão por zero
desvio_features[desvio_features == 0] = 1.0

print(f"Dados utilizados na normalização: {dados_normalizacao.shape}")
print(f"Quantidade de características: {len(colunas_features)}")
print("Normalização calculada com os dados de treinamento.")

In [ ]:
# ============================================================
# CÉLULA 5 — CONSTRUÇÃO DAS SEQUÊNCIAS TEMPORAIS
# ============================================================

# Normalizar as sequências de todas as partidas
partidas_normalizadas = {}

for match_id, sequencia in partidas.items():

    sequencia_normalizada = (
        sequencia - media_features
    ) / desvio_features

    # Concatenar as características das equipes Blue e Red
    sequencia_normalizada = sequencia_normalizada.reshape(
        sequencia_normalizada.shape[0], -1
    )

    partidas_normalizadas[match_id] = sequencia_normalizada.astype(
        np.float32
    )


# Gerar prefixos temporais e seus respectivos rótulos
def criar_exemplos_temporais(ids):

    exemplos = []
    alvos = []
    tamanhos = []

    for match_id in ids:

        sequencia = partidas_normalizadas[match_id]
        vencedor = vencedores[match_id]

        # Gerar prefixos do primeiro até o último instante
        for instante_atual in range(1, sequencia.shape[0] + 1):

            exemplos.append(sequencia[:instante_atual])

            # 0 = Blue | 1 = Red
            alvos.append(0 if vencedor == 100 else 1)

            tamanhos.append(instante_atual)

    return (
        exemplos,
        np.array(alvos, dtype=np.int64),
        np.array(tamanhos, dtype=np.int64)
    )


# Construir os conjuntos de treinamento, validação e teste
X_treino_lista, y_treino, tamanhos_treino = criar_exemplos_temporais(ids_treino)

X_validacao_lista, y_validacao, tamanhos_validacao = criar_exemplos_temporais(ids_validacao)

X_teste_lista, y_teste, tamanhos_teste = criar_exemplos_temporais(ids_teste)


# Resumo dos exemplos gerados
print(f"Exemplos de treinamento: {len(X_treino_lista)}")
print(f"Exemplos de validação: {len(X_validacao_lista)}")
print(f"Exemplos de teste: {len(X_teste_lista)}")

print(f"\nCaracterísticas por instante: {X_treino_lista[0].shape[1]}")
print(f"Formato do primeiro exemplo: {X_treino_lista[0].shape}")

print("\nRótulos: 0 = Blue | 1 = Red")

In [ ]:
# ============================================================
# CÉLULA 6 — PREPARAÇÃO DOS LOTES (BATCHES)
# ============================================================

import tensorflow as tf
import math

BATCH_SIZE = 32
SEMENTE = 42

def criar_dataset(X_lista, y, batch_size=32, shuffle=False):

    def gerador():
        for indice in range(len(X_lista)):

            sequencia = X_lista[indice].astype(np.float32)

            mascara = np.ones(
                sequencia.shape[0],
                dtype=np.float32
            )

            alvo = np.float32(y[indice])

            yield (sequencia, mascara), alvo

    assinatura = (
        (
            tf.TensorSpec(shape=(None, 140), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.float32)
        ),
        tf.TensorSpec(shape=(), dtype=tf.float32)
    )

    dataset = tf.data.Dataset.from_generator(
        gerador,
        output_signature=assinatura
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(X_lista),
            seed=SEMENTE,
            reshuffle_each_iteration=True
        )

    dataset = dataset.padded_batch(
        batch_size,
        padded_shapes=(
            ((None, 140), (None,)),
            ()
        ),
        padding_values=(
            (
                np.float32(0.0),
                np.float32(0.0)
            ),
            np.float32(0.0)
        )
    )

    quantidade_batches = math.ceil(len(X_lista) / batch_size)

    dataset = dataset.apply(
        tf.data.experimental.assert_cardinality(quantidade_batches)
    )

    return dataset.prefetch(tf.data.AUTOTUNE)


# Criar os datasets
dataset_treino = criar_dataset(
    X_treino_lista, y_treino,
    batch_size=BATCH_SIZE,
    shuffle=True
)

dataset_validacao = criar_dataset(
    X_validacao_lista, y_validacao,
    batch_size=BATCH_SIZE
)

dataset_teste = criar_dataset(
    X_teste_lista, y_teste,
    batch_size=BATCH_SIZE
)


# Exibir quantidade de batches
print(f"Batches de treinamento: {math.ceil(len(X_treino_lista) / BATCH_SIZE)}")
print(f"Batches de validação: {math.ceil(len(X_validacao_lista) / BATCH_SIZE)}")
print(f"Batches de teste: {math.ceil(len(X_teste_lista) / BATCH_SIZE)}")

In [ ]:
# ============================================================
# CÉLULA 7 — CONSTRUÇÃO DA CNN1D
# ============================================================

from tensorflow.keras import layers, models, regularizers

# Entrada das sequências temporais
entrada_sequencia = layers.Input(
    shape=(None, 140),
    name="sequencia"
)

# Entrada da máscara temporal
entrada_mascara = layers.Input(
    shape=(None,),
    name="mascara"
)

# Camadas convolucionais
x = layers.Conv1D(
    filters=32,
    kernel_size=3,
    padding="causal",
    activation="relu",
    kernel_regularizer=regularizers.l2(0.001)
)(entrada_sequencia)

x = layers.Conv1D(
    filters=32,
    kernel_size=3,
    padding="causal",
    activation="relu",
    kernel_regularizer=regularizers.l2(0.001)
)(x)

# Pooling médio considerando apenas instantes válidos
mascara_expandida = layers.Reshape((-1, 1))(entrada_mascara)

x = layers.Multiply()([x, mascara_expandida])

soma_temporal = layers.Lambda(
    lambda tensor: tf.reduce_sum(tensor, axis=1)
)(x)

quantidade_instantes = layers.Lambda(
    lambda tensor: tf.reduce_sum(tensor, axis=1)
)(mascara_expandida)

pooling = layers.Lambda(
    lambda tensores: tensores[0] / tf.maximum(tensores[1], 1.0)
)([soma_temporal, quantidade_instantes])

# Camadas de classificação
x = layers.Dense(
    32,
    activation="relu",
    kernel_regularizer=regularizers.l2(0.001)
)(pooling)

x = layers.Dropout(0.4)(x)

saida = layers.Dense(
    1,
    activation="sigmoid",
    name="probabilidade_vitoria"
)(x)

# Construção do modelo
modelo_cnn = models.Model(
    inputs=[entrada_sequencia, entrada_mascara],
    outputs=saida
)

# Exibir arquitetura
modelo_cnn.summary()

In [ ]:
# ============================================================
# CÉLULA 8 — COMPILAÇÃO DO MODELO
# ============================================================

modelo_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),

    loss="binary_crossentropy",

    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

print("Modelo compilado com sucesso.")

In [ ]:
# ============================================================
# CÉLULA 9 — TREINAMENTO DO MODELO
# ============================================================

# Interromper o treinamento quando não houver melhora
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

# Reduzir a taxa de aprendizagem quando necessário
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=0.00001,
    verbose=1
)

# Treinar a CNN1D
historico_cnn = modelo_cnn.fit(
    dataset_treino,
    validation_data=dataset_validacao,
    epochs=50,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

In [ ]:
# ============================================================
# CÉLULA 10 — AVALIAÇÃO DO MODELO
# ============================================================

# Avaliar o modelo no conjunto de teste
resultados_teste = modelo_cnn.evaluate(
    dataset_teste,
    verbose=1
)

# Organizar os resultados
metricas_teste = dict(
    zip(modelo_cnn.metrics_names, resultados_teste)
)

# Exibir as métricas
print("\nRESULTADOS NO CONJUNTO DE TESTE")

for metrica, valor in metricas_teste.items():
    print(f"{metrica}: {valor:.4f}")


In [ ]:
# ============================================================
# CÉLULA 11 — CURVAS DE TREINAMENTO
# ============================================================

import matplotlib.pyplot as plt

# Recuperar o histórico do treinamento
historico = historico_cnn.history

epocas = range(1, len(historico["loss"]) + 1)

# Gráfico da função de perda
plt.figure(figsize=(10, 5))

plt.plot(epocas, historico["loss"], label="Treinamento")
plt.plot(epocas, historico["val_loss"], label="Validação")

plt.title("Função de perda durante o treinamento")
plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# Gráfico da acurácia
plt.figure(figsize=(10, 5))

plt.plot(epocas, historico["accuracy"], label="Treinamento")
plt.plot(epocas, historico["val_accuracy"], label="Validação")

plt.title("Acurácia durante o treinamento")
plt.xlabel("Épocas")
plt.ylabel("Acurácia")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# Gráfico da AUC
plt.figure(figsize=(10, 5))

plt.plot(epocas, historico["auc"], label="Treinamento")
plt.plot(epocas, historico["val_auc"], label="Validação")

plt.title("AUC durante o treinamento")
plt.xlabel("Épocas")
plt.ylabel("AUC")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:

# ============================================================
# CÉLULA 12 — CNN1D AJUSTADA PARA REDUÇÃO DO OVERFITTING
# ============================================================

from tensorflow.keras import layers, models, regularizers

# Entrada das sequências temporais
entrada_sequencia = layers.Input(
    shape=(None, 140),
    name="sequencia"
)

# Entrada da máscara temporal
entrada_mascara = layers.Input(
    shape=(None,),
    name="mascara"
)

# Camadas convolucionais com menor quantidade de filtros
x = layers.Conv1D(
    filters=16,
    kernel_size=3,
    padding="causal",
    activation="relu",
    kernel_regularizer=regularizers.l2(0.002)
)(entrada_sequencia)

x = layers.Conv1D(
    filters=16,
    kernel_size=3,
    padding="causal",
    activation="relu",
    kernel_regularizer=regularizers.l2(0.002)
)(x)

# Pooling médio considerando apenas instantes válidos
mascara_expandida = layers.Reshape((-1, 1))(entrada_mascara)

x = layers.Multiply()([x, mascara_expandida])

soma_temporal = layers.Lambda(
    lambda tensor: tf.reduce_sum(tensor, axis=1)
)(x)

quantidade_instantes = layers.Lambda(
    lambda tensor: tf.reduce_sum(tensor, axis=1)
)(mascara_expandida)

pooling = layers.Lambda(
    lambda tensores: tensores[0] / tf.maximum(tensores[1], 1.0)
)([soma_temporal, quantidade_instantes])

# Camada densa com regularização
x = layers.Dense(
    16,
    activation="relu",
    kernel_regularizer=regularizers.l2(0.002)
)(pooling)

# Aumentar o dropout para reduzir o sobreajuste
x = layers.Dropout(0.5)(x)

# Saída: probabilidade de vitória da equipe Red
saida = layers.Dense(
    1,
    activation="sigmoid",
    name="probabilidade_vitoria"
)(x)

# Construção do novo modelo
modelo_cnn = models.Model(
    inputs=[entrada_sequencia, entrada_mascara],
    outputs=saida
)

# Exibir arquitetura
modelo_cnn.summary()


In [ ]:

# ============================================================
# CÉLULA 13 — AVALIAÇÃO DETALHADA DO MODELO
# ============================================================

resultados_teste = modelo_cnn.evaluate(
    dataset_teste,
    return_dict=True,
    verbose=1
)

print("\nRESULTADOS NO CONJUNTO DE TESTE")

for metrica, valor in resultados_teste.items():
    print(f"{metrica}: {valor:.4f}")


In [ ]:
# ============================================================
# CÉLULA 14 — MATRIZ DE CONFUSÃO E RELATÓRIO DE CLASSIFICAÇÃO
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Gerar probabilidades de previsão
probabilidades = modelo_cnn.predict(
    dataset_teste,
    verbose=1
).ravel()

# Converter probabilidades em classes
y_pred = (probabilidades >= 0.5).astype(int)

# Calcular matriz de confusão
matriz_confusao = np.zeros((2, 2), dtype=int)

for real, previsto in zip(y_teste, y_pred):
    matriz_confusao[int(real), int(previsto)] += 1

# Exibir matriz de confusão
fig, ax = plt.subplots(figsize=(7, 5))

imagem = ax.imshow(matriz_confusao, cmap="Blues")

ax.set_title("Matriz de Confusão — CNN1D")
ax.set_xlabel("Classe prevista")
ax.set_ylabel("Classe real")

ax.set_xticks([0, 1])
ax.set_xticklabels(["Blue", "Red"])

ax.set_yticks([0, 1])
ax.set_yticklabels(["Blue", "Red"])

for i in range(2):
    for j in range(2):
        ax.text(
            j, i,
            matriz_confusao[i, j],
            ha="center",
            va="center",
            fontsize=12
        )

fig.colorbar(imagem, ax=ax)
plt.tight_layout()
plt.show()

# Calcular métricas por classe
print("\nRELATÓRIO DE CLASSIFICAÇÃO")

nomes_classes = ["Blue", "Red"]

for classe in range(2):

    verdadeiros_positivos = matriz_confusao[classe, classe]

    falsos_positivos = (
        matriz_confusao[:, classe].sum() - verdadeiros_positivos
    )

    falsos_negativos = (
        matriz_confusao[classe, :].sum() - verdadeiros_positivos
    )

    precisao = (
        verdadeiros_positivos / (verdadeiros_positivos + falsos_positivos)
        if (verdadeiros_positivos + falsos_positivos) > 0 else 0
    )

    recall = (
        verdadeiros_positivos / (verdadeiros_positivos + falsos_negativos)
        if (verdadeiros_positivos + falsos_negativos) > 0 else 0
    )

    f1 = (
        2 * precisao * recall / (precisao + recall)
        if (precisao + recall) > 0 else 0
    )

    print(f"\nClasse: {nomes_classes[classe]}")
    print(f"Precisão: {precisao:.4f}")
    print(f"Recall:   {recall:.4f}")
    print(f"F1-score: {f1:.4f}")

# Acurácia geral
acuracia = np.mean(y_teste == y_pred)

print(f"\nAcurácia geral: {acuracia:.4f}")

In [ ]:
# ============================================================
# CÉLULA 15 — FUNÇÃO DE PREVISÃO DO VENCEDOR
# ============================================================

def prever_vencedor(serie_da_partida):

    # Verificar o formato de entrada
    if not isinstance(serie_da_partida, pd.DataFrame):
        raise TypeError("A entrada deve ser um DataFrame.")

    dados = serie_da_partida.copy()

    # Verificar se o arquivo contém apenas uma partida
    if "match_id" in dados.columns:
        if dados["match_id"].nunique() != 1:
            raise ValueError(
                "O DataFrame deve conter apenas uma partida."
            )

    # Verificar as colunas obrigatórias
    colunas_obrigatorias = (
        ["timestep", "teamId"] + colunas_features
    )

    colunas_ausentes = [
        coluna for coluna in colunas_obrigatorias
        if coluna not in dados.columns
    ]

    if colunas_ausentes:
        raise ValueError(
            f"Colunas ausentes: {colunas_ausentes}"
        )

    # Organizar os registros cronologicamente
    dados = dados.sort_values(
        ["timestep", "teamId"]
    )

    sequencia = []

    # Construir a sequência temporal
    for timestep, instante in dados.groupby("timestep", sort=True):

        equipes_presentes = instante["teamId"].tolist()

        # Verificar a presença das duas equipes
        if (
            len(equipes_presentes) != 2
            or sorted(equipes_presentes) != [100, 200]
        ):
            raise ValueError(
                f"Timestep {timestep}: esperado um registro "
                "para Blue (100) e outro para Red (200)."
            )

        # Garantir a ordem Blue e Red
        instante_ordenado = (
            instante
            .set_index("teamId")
            .reindex([100, 200])
        )

        # Extrair as características
        matriz_instante = instante_ordenado[
            colunas_features
        ].to_numpy(dtype=np.float32)

        # Verificar valores ausentes ou inválidos
        if not np.isfinite(matriz_instante).all():
            raise ValueError(
                f"Existem valores ausentes ou inválidos "
                f"no timestep {timestep}."
            )

        sequencia.append(matriz_instante)

    if not sequencia:
        raise ValueError("A partida não possui dados para previsão.")

    # Formato: (timesteps, 2 equipes, características)
    sequencia = np.stack(sequencia)

    # Aplicar a normalização do treinamento
    sequencia = (
        sequencia - media_features
    ) / desvio_features

    # Concatenar Blue e Red
    sequencia = sequencia.reshape(
        sequencia.shape[0], -1
    ).astype(np.float32)

    # Preparar entrada para a CNN1D
    entrada = np.expand_dims(sequencia, axis=0)

    mascara = np.ones(
        (1, sequencia.shape[0]),
        dtype=np.float32
    )

    # Obter probabilidade da classe Red
    probabilidade_red = float(
        modelo_cnn.predict(
            [entrada, mascara],
            verbose=0
        )[0][0]
    )

    # Calcular probabilidade da classe Blue
    probabilidade_blue = 1.0 - probabilidade_red

    # Identificar a equipe prevista
    equipe_prevista = (
        "Red" if probabilidade_red >= 0.5 else "Blue"
    )

    return {
        "equipe_prevista": equipe_prevista,
        "probabilidade": round(
            max(probabilidade_blue, probabilidade_red) * 100, 2
        ),
        "probabilidade_blue": round(
            probabilidade_blue * 100, 2
        ),
        "probabilidade_red": round(
            probabilidade_red * 100, 2
        ),
        "timesteps_utilizados": sequencia.shape[0]
    }


print("Função prever_vencedor() definida com sucesso.")

In [ ]:
# ============================================================
# CÉLULA 16 — TESTE DA FUNÇÃO DE PREVISÃO
# ============================================================

# Carregar uma partida
serie_da_partida = pd.read_excel(
    "../data/arquivo_partida.xlsx"
)

# Realizar previsão
resultado = prever_vencedor(serie_da_partida)

# Exibir resultado
print("RESULTADO DA PREVISÃO")
print(f"Equipe prevista: {resultado['equipe_prevista']}")
print(f"Probabilidade: {resultado['probabilidade']}%")
print(f"Probabilidade Blue: {resultado['probabilidade_blue']}%")
print(f"Probabilidade Red: {resultado['probabilidade_red']}%")
print(f"Timesteps utilizados: {resultado['timesteps_utilizados']}")

In [ ]:
# ============================================================
# CÉLULA 17 — TESTE DA FUNÇÃO DE PREVISÃO POR MINUTO
# ============================================================

import time

# Intervalo entre as previsões (em segundos)
INTERVALO_SEGUNDOS = 0.5

for timestep in sorted(serie_da_partida["timestep"].unique()):

    inicio = time.perf_counter()

    partida_parcial = serie_da_partida[
        serie_da_partida["timestep"] <= timestep
    ]

    resultado = prever_vencedor(partida_parcial)

    tempo_execucao = time.perf_counter() - inicio

    print(
        f"Timestep {timestep} | "
        f"Previsão: {resultado['equipe_prevista']} | "
        f"Blue: {resultado['probabilidade_blue']}% | "
        f"Red: {resultado['probabilidade_red']}% | "
        f"Execução: {tempo_execucao:.3f}s"
    )

    # Aguardar antes da próxima previsão, exceto no último ciclo
    if timestep != serie_da_partida["timestep"].max():
        time.sleep(INTERVALO_SEGUNDOS)

In [ ]:
# ============================================================
# CÉLULA 18 — VISUALIZAÇÃO DAS VARIÁVEIS DA PARTIDA
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Variáveis que terão gráficos gerados
# VARIAVEIS_GRAFICOS = [
#     "diff_totalGold",
#     "diff_xp",
#     # "totalDamageDealt",
# ]

VARIAVEIS_GRAFICOS = [
    "diff_totalGold",
    "diff_xp",
    "xp",
    "kills",
    "dragons",
    "elder_dragons",
    "barons"

]

# ------------------------------------------------------------
# 1. Preparar os dados
# ------------------------------------------------------------

dados_graficos = serie_da_partida.copy()

dados_graficos = dados_graficos.sort_values(
    ["timestep", "teamId"]
)

timesteps = sorted(dados_graficos["timestep"].unique())

# ------------------------------------------------------------
# 2. Gerar um gráfico para cada variável selecionada
# ------------------------------------------------------------

for variavel in VARIAVEIS_GRAFICOS:

    if variavel not in dados_graficos.columns:
        print(f"Variável não encontrada: {variavel}")
        continue

    # --------------------------------------------------------
    # GRÁFICO DE DIFERENÇA — LINHA
    # --------------------------------------------------------

    if variavel.startswith("diff_"):

        valores = (
            dados_graficos[
                ["timestep", variavel]
            ]
            .drop_duplicates(subset="timestep")
            .sort_values("timestep")
        )

        x = valores["timestep"].to_numpy()
        y = valores[variavel].to_numpy(dtype=float)

        fig, ax = plt.subplots(figsize=(12, 4))

        # Área positiva: Blue
        ax.fill_between(
            x, y, 0,
            where=(y >= 0),
            interpolate=True,
            color="steelblue",
            alpha=0.55
        )

        # Área negativa: Red
        ax.fill_between(
            x, y, 0,
            where=(y < 0),
            interpolate=True,
            color="indianred",
            alpha=0.55
        )

        # Linha azul nos valores positivos
        ax.plot(
            x, np.where(y >= 0, y, np.nan),
            color="steelblue",
            linewidth=2
        )

        # Linha vermelha nos valores negativos
        ax.plot(
            x, np.where(y < 0, y, np.nan),
            color="indianred",
            linewidth=2
        )

        # Linha horizontal do zero
        ax.axhline(
            0,
            color="gray",
            linewidth=0.8
        )

        # Grade vertical em cada timestep
        ax.set_xticks(x)
        ax.grid(
            axis="both",
            alpha=0.2,
            linestyle="-"
        )

        ax.set_title(variavel, fontsize=14)
        ax.set_xlabel("Timestep")
        ax.set_ylabel("Diferença")

        plt.tight_layout()
        plt.show()

    # --------------------------------------------------------
    # GRÁFICO DE VARIÁVEL CORRENTE — BARRAS
    # --------------------------------------------------------

    else:

        dados_blue = (
            dados_graficos[dados_graficos["teamId"] == 100]
            .sort_values("timestep")
        )

        dados_red = (
            dados_graficos[dados_graficos["teamId"] == 200]
            .sort_values("timestep")
        )

        # Garantir correspondência entre os timesteps
        dados_blue = dados_blue.set_index("timestep").reindex(timesteps)
        dados_red = dados_red.set_index("timestep").reindex(timesteps)

        valores_blue = dados_blue[variavel].to_numpy(dtype=float)
        valores_red = dados_red[variavel].to_numpy(dtype=float)

        posicoes = np.arange(len(timesteps))

        largura = 0.36
        deslocamento = 0.20

        fig, ax = plt.subplots(figsize=(12, 4))

        # Barras Blue
        ax.bar(
            posicoes - deslocamento,
            valores_blue,
            width=largura,
            color="steelblue",
            alpha=0.65,
            edgecolor="steelblue",
            label="Blue"
        )

        # Barras Red
        ax.bar(
            posicoes + deslocamento,
            valores_red,
            width=largura,
            color="indianred",
            alpha=0.65,
            edgecolor="indianred",
            label="Red"
        )

        ax.set_title(variavel, fontsize=14)
        ax.set_xlabel("Timestep")
        ax.set_ylabel("Valor")
        ax.set_xticks(posicoes)
        ax.set_xticklabels(timesteps)
        ax.legend()
        ax.grid(axis="y", alpha=0.2)

        plt.tight_layout()
        plt.show()

In [ ]:
# ============================================================
# CÉLULA 19 — RELATÓRIO TEXTUAL: METODOLOGIA E RESULTADOS
# ============================================================

import numpy as np

# ------------------------------------------------------------
# 1. CONFIGURAÇÕES
# ------------------------------------------------------------

# Nomes possíveis para o conjunto de teste
nomes_datasets_teste = [
    "dataset_teste",
    "ds_teste",
    "teste_dataset",
    "dados_teste"
]

# Arquitetura utilizada
FILTROS_CNN = 16
TAMANHO_KERNEL = 3
UNIDADES_DENSAS = 16
DROPOUT = 0.5
L2 = 0.002
TAXA_APRENDIZADO = 0.001
TAMANHO_LOTE = 32

# ------------------------------------------------------------
# 2. FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def obter_variavel(nomes):
    """Retorna a primeira variável existente no notebook."""
    for nome in nomes:
        if nome in globals():
            return globals()[nome]
    return None


def formatar_numero(valor, casas=4):
    if valor is None:
        return "não disponível"
    return f"{valor:.{casas}f}".replace(".", ",")


def formatar_percentual(valor):
    if valor is None:
        return "não disponível"
    return f"{valor * 100:.2f}%".replace(".", ",")


# ------------------------------------------------------------
# 3. INFORMAÇÕES DO CONJUNTO DE DADOS
# ------------------------------------------------------------

quantidade_features = len(colunas_features)

ids_treino = obter_variavel(["ids_treino"])
ids_validacao = obter_variavel(["ids_validacao"])
ids_teste = obter_variavel(["ids_teste"])

quantidade_treino = len(ids_treino) if ids_treino is not None else None
quantidade_validacao = len(ids_validacao) if ids_validacao is not None else None
quantidade_teste = len(ids_teste) if ids_teste is not None else None

quantidade_partidas = (
    quantidade_treino + quantidade_validacao + quantidade_teste
    if None not in [
        quantidade_treino,
        quantidade_validacao,
        quantidade_teste
    ]
    else None
)

# Quantidade de exemplos temporais, quando disponível
X_treino = obter_variavel(["X_treino", "x_treino"])
X_validacao = obter_variavel(["X_validacao", "x_validacao"])
X_teste = obter_variavel(["X_teste", "x_teste"])

# ------------------------------------------------------------
# 4. INFORMAÇÕES DO TREINAMENTO
# ------------------------------------------------------------

historico = historico_cnn.history

epocas_executadas = len(historico.get("loss", []))

melhor_epoca = None
melhor_val_loss = None

if "val_loss" in historico and len(historico["val_loss"]) > 0:
    melhor_epoca = int(np.argmin(historico["val_loss"])) + 1
    melhor_val_loss = float(np.min(historico["val_loss"]))

ultima_epoca = epocas_executadas - 1

acuracia_treino = (
    historico["accuracy"][ultima_epoca]
    if "accuracy" in historico else None
)

acuracia_validacao = (
    historico["val_accuracy"][ultima_epoca]
    if "val_accuracy" in historico else None
)

# ------------------------------------------------------------
# 5. AVALIAÇÃO DO CONJUNTO DE TESTE
# ------------------------------------------------------------

dataset_teste_relatorio = obter_variavel(nomes_datasets_teste)

metricas_teste = None
matriz_confusao = None
metricas_classes = None

if dataset_teste_relatorio is not None:

    # Avaliação geral
    metricas_teste = modelo_cnn.evaluate(
        dataset_teste_relatorio,
        return_dict=True,
        verbose=0
    )

    # Previsões para calcular a matriz de confusão
    previsoes = modelo_cnn.predict(
        dataset_teste_relatorio,
        verbose=0
    ).reshape(-1)

    valores_reais = []

    for lote in dataset_teste_relatorio:
        rotulos = lote[1].numpy().reshape(-1)
        valores_reais.extend(rotulos)

    valores_reais = np.asarray(valores_reais).astype(int)
    valores_previstos = (previsoes >= 0.5).astype(int)

    # Matriz de confusão
    matriz_confusao = np.zeros((2, 2), dtype=int)

    for real, previsto in zip(valores_reais, valores_previstos):
        matriz_confusao[real, previsto] += 1

    # Métricas por classe
    metricas_classes = {}

    for classe, nome in [(0, "Blue"), (1, "Red")]:

        verdadeiros_positivos = matriz_confusao[classe, classe]
        falsos_positivos = matriz_confusao[1 - classe, classe]
        falsos_negativos = matriz_confusao[classe, 1 - classe]

        precisao = (
            verdadeiros_positivos /
            (verdadeiros_positivos + falsos_positivos)
            if verdadeiros_positivos + falsos_positivos > 0
            else 0
        )

        recall = (
            verdadeiros_positivos /
            (verdadeiros_positivos + falsos_negativos)
            if verdadeiros_positivos + falsos_negativos > 0
            else 0
        )

        f1 = (
            2 * precisao * recall / (precisao + recall)
            if precisao + recall > 0
            else 0
        )

        metricas_classes[nome] = {
            "precisao": precisao,
            "recall": recall,
            "f1": f1
        }

# ------------------------------------------------------------
# 6. TEXTO DA METODOLOGIA
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("3.4. MODELAGEM PREDITIVA COM REDES NEURAIS CONVOLUCIONAIS")
print("=" * 80)

texto_metodologia = f"""
Para a tarefa de predição do resultado das partidas, foi desenvolvida
uma arquitetura de rede neural convolucional unidimensional (CNN1D),
implementada utilizando a biblioteca TensorFlow, por meio da API
Keras. A escolha dessa arquitetura considerou a natureza temporal
dos dados, uma vez que as variáveis do jogo são registradas em
diferentes instantes e apresentam evolução ao longo da partida.

Inicialmente, os registros foram organizados cronologicamente por
partida e timestep. Como cada instante possuía duas observações,
uma referente à equipe Blue (teamId = 100) e outra à equipe Red
(teamId = 200), os registros foram reorganizados para manter essa
ordem. As {quantidade_features} variáveis de cada equipe foram
concatenadas, formando um vetor de {quantidade_features * 2}
características por instante temporal.

Na etapa de preparação, foram identificados valores ausentes no
último timestep das partidas. Para manter a consistência dos dados
utilizados na modelagem, esse instante foi removido durante a
organização das séries temporais. Em seguida, as características
foram normalizadas por padronização, utilizando a média e o desvio
padrão calculados exclusivamente sobre o conjunto de treinamento,
evitando a utilização de informações dos conjuntos de validação e
teste no processo de normalização.

A divisão dos dados foi realizada por partida, de modo que os
registros pertencentes a uma mesma partida permanecessem em apenas
um dos subconjuntos. Foram utilizados {quantidade_treino if quantidade_treino is not None else "N/D"}
jogos para treinamento, {quantidade_validacao if quantidade_validacao is not None else "N/D"}
para validação e {quantidade_teste if quantidade_teste is not None else "N/D"}
para teste. A divisão seguiu as proporções de 70%, 15% e 15%,
respectivamente, com estratificação baseada na equipe vencedora
e utilização de semente aleatória fixa.

Para ampliar a quantidade de exemplos temporais, cada partida foi
convertida em uma sequência de exemplos cumulativos. Dessa forma,
foram consideradas sequências correspondentes aos diferentes
prefixos temporais disponíveis, permitindo que o modelo aprendesse
a identificar padrões associados ao resultado a partir de diferentes
momentos da partida. Como as sequências poderiam apresentar
comprimentos distintos, foi utilizado preenchimento de sequências
(padding), acompanhado de máscaras para impedir que os valores
adicionados artificialmente influenciassem o cálculo das
representações temporais.

A arquitetura foi composta por duas camadas convolucionais
unidimensionais, cada uma com {FILTROS_CNN} filtros e tamanho de
kernel {TAMANHO_KERNEL}, utilizando padding causal e função de
ativação ReLU. A regularização L2, com coeficiente {L2},
foi empregada nas camadas para reduzir o risco de sobreajuste.
Após as convoluções, foi aplicado pooling médio com tratamento
da máscara temporal, seguido por uma camada densa de
{UNIDADES_DENSAS} unidades, também com ativação ReLU, e uma camada
de dropout de {DROPOUT}. A camada de saída possui um neurônio com
ativação sigmoid, responsável por estimar a probabilidade da
classe Red. Consequentemente, a probabilidade complementar
representa a classe Blue.

O treinamento foi realizado com o otimizador Adam, taxa de
aprendizado inicial de {TAXA_APRENDIZADO}, função de perda
binary cross-entropy e lote de {TAMANHO_LOTE} exemplos. Foram
utilizados mecanismos de Early Stopping e redução adaptativa da
taxa de aprendizado, com o objetivo de acompanhar o desempenho
na validação e limitar a continuidade do treinamento quando não
houvesse melhoria na função de perda.
"""

print(texto_metodologia)

# ------------------------------------------------------------
# 7. TEXTO DOS RESULTADOS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("4.4. DESEMPENHO DO MODELO DE PREDIÇÃO")
print("=" * 80)

texto_resultados = f"""
O treinamento da arquitetura CNN1D foi executado por
{epocas_executadas} épocas. Considerando o acompanhamento da
função de perda no conjunto de validação, a melhor época foi
{melhor_epoca if melhor_epoca is not None else "não identificada"},
com valor de perda igual a
{formatar_numero(melhor_val_loss)}. Ao longo do treinamento,
foram monitoradas métricas de desempenho como acurácia, AUC,
precisão e recall, permitindo acompanhar tanto a capacidade
discriminativa quanto o comportamento das classificações.
"""

if acuracia_treino is not None:
    texto_resultados += f"""
Na última época executada, a acurácia observada no treinamento
foi de {formatar_percentual(acuracia_treino)}."""

if acuracia_validacao is not None:
    texto_resultados += f"""
No conjunto de validação, a acurácia registrada na mesma época
foi de {formatar_percentual(acuracia_validacao)}."""

if metricas_teste is not None:

    texto_resultados += f"""

A avaliação no conjunto de teste apresentou acurácia de
{formatar_percentual(metricas_teste.get("accuracy"))},
AUC de {formatar_numero(metricas_teste.get("auc"))},
precisão de {formatar_percentual(metricas_teste.get("precision"))}
e recall de {formatar_percentual(metricas_teste.get("recall"))}.
A função de perda no teste foi de
{formatar_numero(metricas_teste.get("loss"))}.

A matriz de confusão permitiu examinar o comportamento do
classificador para as duas classes. Para a equipe Blue, foram
obtidos precisão de
{formatar_percentual(metricas_classes["Blue"]["precisao"])},
recall de {formatar_percentual(metricas_classes["Blue"]["recall"])}
e F1-score de {formatar_numero(metricas_classes["Blue"]["f1"])}.
Para a equipe Red, os valores foram, respectivamente,
{formatar_percentual(metricas_classes["Red"]["precisao"])},
{formatar_percentual(metricas_classes["Red"]["recall"])} e
{formatar_numero(metricas_classes["Red"]["f1"])}.

A matriz de confusão apresentou os seguintes valores, considerando
as linhas como classes reais e as colunas como classes previstas:
"""

    texto_resultados += f"""

                 Predito Blue    Predito Red
Real Blue             {matriz_confusao[0, 0]}              {matriz_confusao[0, 1]}
Real Red              {matriz_confusao[1, 0]}              {matriz_confusao[1, 1]}
"""

else:
    texto_resultados += """

As métricas finais do conjunto de teste não foram calculadas
automaticamente, pois não foi localizado o objeto do dataset de
teste. Verifique a variável utilizada na criação desse conjunto
e inclua seu nome na lista nomes_datasets_teste, no início desta
célula.
"""

texto_resultados += """

Para complementar a análise quantitativa, recomenda-se apresentar
a evolução das métricas e da função de perda durante o treinamento,
permitindo observar o comportamento do modelo nos conjuntos de
treinamento e validação [Figura 1 – Curvas de acurácia e perda
durante o treinamento].

A distribuição dos acertos e erros de classificação pode ser
apresentada por meio da matriz de confusão [Figura 2 – Matriz de
confusão do modelo no conjunto de teste]. Essa representação
permite identificar possíveis diferenças no desempenho para as
classes Blue e Red.

Por fim, a análise das previsões realizadas a partir de diferentes
prefixos temporais permite observar como as probabilidades
atribuídas às equipes se modificam conforme novos instantes da
partida são incorporados [Figura 3 – Evolução das probabilidades
de vitória das equipes ao longo dos timesteps].
"""

print(texto_resultados)

# ------------------------------------------------------------
# 8. RESUMO NUMÉRICO PARA CONFERÊNCIA
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RESUMO DOS DADOS E DO TREINAMENTO")
print("=" * 80)

print(f"Variáveis por equipe: {quantidade_features}")
print(f"Variáveis por timestep após concatenação: {quantidade_features * 2}")
print(f"Partidas totais: {quantidade_partidas}")
print(f"Partidas de treinamento: {quantidade_treino}")
print(f"Partidas de validação: {quantidade_validacao}")
print(f"Partidas de teste: {quantidade_teste}")
print(f"Épocas executadas: {epocas_executadas}")
print(f"Melhor época: {melhor_epoca}")

if metricas_teste is not None:
    print(f"Exemplos avaliados no teste: {len(valores_reais)}")
    print(f"Acurácia no teste: {formatar_percentual(metricas_teste.get('accuracy'))}")
    print(f"AUC no teste: {formatar_numero(metricas_teste.get('auc'))}")